# Vision Transformer (ViT-B/16) Training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchinfo', 'fvcore'], check=True)

In [ ]:
import os, zipfile, time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
from torchvision.models.vision_transformer import vit_b_16
from torch.amp import autocast, GradScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from fvcore.nn import FlopCountAnalysis

# Check GPU before proceeding
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not available! Go to Runtime > Change runtime type > T4 GPU')

zip_path     = '/content/drive/MyDrive/archive.zip'
dataset_root = '/content/fruits'

if not os.path.exists(dataset_root):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dataset_root)
    print('Dataset extracted!')

subfolders   = [f.name for f in os.scandir(dataset_root) if f.is_dir()]
dataset_path = os.path.join(dataset_root, subfolders[0]) if len(subfolders) == 1 else dataset_root
print('Dataset path:', dataset_path)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_data = datasets.ImageFolder(root=dataset_path, transform=train_transform)
eval_data  = datasets.ImageFolder(root=dataset_path, transform=eval_transform)
total      = len(train_data)
train_size = int(0.70 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

indices       = torch.randperm(total, generator=torch.Generator().manual_seed(42)).tolist()
train_dataset = Subset(train_data, indices[:train_size])
val_dataset   = Subset(eval_data,  indices[train_size:train_size + val_size])
test_dataset  = Subset(eval_data,  indices[train_size + val_size:])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=2)

num_classes  = len(train_data.classes)
print('Classes:', train_data.classes)
print(f'Train: {train_size} | Val: {val_size} | Test: {test_size}')

In [ ]:
device = torch.device('cuda')
print('Device:', device)

model = vit_b_16(weights='IMAGENET1K_V1')
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
flops = FlopCountAnalysis(model, torch.randn(1, 3, 224, 224).to(device))
flops.unsupported_ops_warnings(False)
total_flops_g = flops.total() / 1e9
print(f'Total Parameters : {total_params:.2f}M')
print(f'Total FLOPs      : {total_flops_g:.2f}G')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
scaler    = GradScaler()

num_epochs       = 30
patience         = 5
best_val_loss    = float('inf')
patience_counter = 0
best_epoch       = 0
epoch_times      = []

print('Training Started...\n')
for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    epoch_start = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        _, predicted  = torch.max(outputs, 1)
        correct      += (predicted == labels).sum().item()
        total        += labels.size(0)

    scheduler.step()
    epoch_time = time.time() - epoch_start
    train_loss = running_loss / len(train_loader)
    train_acc  = 100 * correct / total
    epoch_times.append(epoch_time)

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            v_loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            v_correct   += (predicted == labels).sum().item()
            v_total     += labels.size(0)

    val_loss = v_loss / len(val_loader)
    val_acc  = 100 * v_correct / v_total
    print(f'Epoch [{epoch+1:02d}/{num_epochs}] Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%  Time: {epoch_time:.1f}s')

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = epoch + 1
        patience_counter = 0
        torch.save(model.state_dict(), '/content/drive/MyDrive/vit_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1} (best: {best_epoch})')
            break

avg_epoch_time = sum(epoch_times) / len(epoch_times)
print(f'\nTraining Complete! Best epoch: {best_epoch}')
print(f'Avg training time per epoch: {avg_epoch_time:.2f}s')

In [ ]:
model.load_state_dict(torch.load('/content/drive/MyDrive/vit_best.pt', map_location=device))
torch.save(model.state_dict(), '/content/drive/MyDrive/vit_weights.pt')
model_size_mb = os.path.getsize('/content/drive/MyDrive/vit_weights.pt') / (1024 * 1024)
print(f'Best model saved | Size: {model_size_mb:.2f} MB')

model.eval()
all_preds, all_labels, inference_times = [], [], []
test_loss, correct, total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        t0      = time.time()
        outputs = model(images)
        inference_times.append((time.time() - t0) * 1000)
        loss      = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct     += (predicted == labels).sum().item()
        total       += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

macro_p  = precision_score(all_labels, all_preds, average='macro')
macro_r  = recall_score   (all_labels, all_preds, average='macro')
macro_f1 = f1_score       (all_labels, all_preds, average='macro')
avg_inf  = sum(inference_times) / len(inference_times)

print('\n' + '='*60)
print('         VISION TRANSFORMER (ViT) - RESULTS SUMMARY')
print('='*60)
print(f'  Parameters          : {total_params:.2f} M')
print(f'  Model Size          : {model_size_mb:.2f} MB')
print(f'  FLOPs               : {total_flops_g:.2f} G')
print(f'  Best Epoch          : {best_epoch}')
print(f'  Avg Epoch Time      : {avg_epoch_time:.2f} s')
print(f'  Test Loss           : {test_loss/len(test_loader):.4f}')
print(f'  Test Accuracy       : {100*correct/total:.2f} %')
print(f'  Macro Precision     : {macro_p:.4f}')
print(f'  Macro Recall        : {macro_r:.4f}')
print(f'  Macro F1-Score      : {macro_f1:.4f}')
print(f'  Avg Inference Time  : {avg_inf:.2f} ms/batch')
print('='*60)
print('\nPer-Class Classification Report:')
print(classification_report(all_labels, all_preds, target_names=train_data.classes))
print('Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))